# Prior Sensitivity Analysis on Province Model

In [14]:
from pathlib import Path
import os, stat
from tqdm import tqdm

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel

In [15]:
project_root = Path('..').resolve()
stan_path = project_root / 'stan' / 'logistic_province_prior_sensitivity.stan'
out_root = Path.cwd() / 'stan_output' / 'sensitivity_province'
out_root.mkdir(parents=True, exist_ok=True)

def compile_model(stan_path):
    print('Compiling', stan_path.name)
    mdl = CmdStanModel(stan_file=str(stan_path))
    mode = os.stat(mdl.exe_file).st_mode
    os.chmod(mdl.exe_file, mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    return mdl

mdl_sens = compile_model(stan_path)

Compiling logistic_province_prior_sensitivity.stan


In [16]:
df = pd.read_csv('../data/fire_db.csv')
df_proc = df.dropna(subset=['any_evacuation', 'log_fire_size_ha', 'fn_indicator', 'province']).copy()
prov_cat = pd.Categorical(df_proc['province'])
df_proc['prov_idx'] = prov_cat.codes + 1

base_data = {
    'N': len(df_proc),
    'P': len(prov_cat.categories),
    'province': df_proc['prov_idx'].astype(int).tolist(),
    'y': df_proc['any_evacuation'].astype(int).tolist(),
    'log_fire_size': df_proc['log_fire_size_ha'].astype(float).tolist(),
    'fn_indicator': df_proc['fn_indicator'].astype(int).tolist(),
}

print('N =', base_data['N'], 'P =', base_data['P'])

N = 15203 P = 12


In [17]:
baseline = {
    'sigma_alpha': 2.0,
    'sigma_prov': 1.0,
    'sigma_beta_log_fire_size': 1.0,
    'sigma_beta_fn': 1.0,
}

configs = [
    {'name': 'half_all', **{k: v * 0.5 for k, v in baseline.items()}},
    {'name': 'baseline', **baseline},
    {'name': 'double_all', **{k: v * 2.0 for k, v in baseline.items()}},
]

grid = pd.DataFrame(configs)
grid

,name,sigma_alpha,sigma_prov,sigma_beta_log_fire_size,sigma_beta_fn
0,half_all,1.0,0.5,0.5,0.5
1,baseline,2.0,1.0,1.0,1.0
2,double_all,4.0,2.0,2.0,2.0


In [ ]:
results = []
loo_dict = {}
summary_dict = {}
core_posterior_vars = ['alpha', 'beta_log_fire_size', 'beta_fn']

def _compact_idata(idata):
    idata.posterior = idata.posterior[core_posterior_vars]
    idata.log_likelihood = idata.log_likelihood[['log_lik']]
    for grp in ['posterior_predictive', 'predictions', 'prior', 'prior_predictive', 'sample_stats']:
        if hasattr(idata, grp):
            delattr(idata, grp)
    return idata

def _as_float_or_nan(x):
    try:
        return float(x)
    except Exception:
        return float('nan')

def _extract_elpd_and_se(loo_obj):
    # ArviZ/ArviZ-Stats compatibility across versions
    elpd = None
    for attr in ['elpd_loo', 'elpd', 'loo', 'estimate']:
        if hasattr(loo_obj, attr):
            elpd = getattr(loo_obj, attr)
            break
    if elpd is None and hasattr(loo_obj, 'to_dict'):
        d = loo_obj.to_dict()
        for key in ['elpd_loo', 'elpd', 'loo', 'estimate']:
            if key in d:
                elpd = d[key]
                break

    se = None
    for attr in ['se', 'elpd_se', 'standard_error']:
        if hasattr(loo_obj, attr):
            se = getattr(loo_obj, attr)
            break
    if se is None and hasattr(loo_obj, 'to_dict'):
        d = loo_obj.to_dict()
        for key in ['se', 'elpd_se', 'standard_error']:
            if key in d:
                se = d[key]
                break

    return _as_float_or_nan(elpd), _as_float_or_nan(se)

for cfg in tqdm(configs):
    name = cfg['name']
    print(f'Running {name}')

    data = dict(base_data)
    data['sigma_alpha'] = cfg['sigma_alpha']
    data['sigma_prov'] = cfg['sigma_prov']
    data['sigma_beta_log_fire_size'] = cfg['sigma_beta_log_fire_size']
    data['sigma_beta_fn'] = cfg['sigma_beta_fn']

    run_dir = out_root / name
    run_dir.mkdir(parents=True, exist_ok=True)

    idata_path = run_dir / 'idata_compact.nc'
    legacy_idata_path = run_dir / 'idata.nc'

    if idata_path.exists():
        print(f'Loading cached inference data for {name} from {idata_path.name}')
        idata = az.from_netcdf(str(idata_path))
    elif legacy_idata_path.exists():
        print(f'Converting legacy cache for {name} to compact format')
        idata = az.from_netcdf(str(legacy_idata_path))
        idata = _compact_idata(idata)
        idata.to_netcdf(str(idata_path))
    else:
        fit = mdl_sens.sample(
            data=data,
            chains=4,
            parallel_chains=min(4, os.cpu_count() or 1),
            iter_warmup=500,
            iter_sampling=1000,
            seed=20260419,
            adapt_delta=0.95,
            show_console=False,
            output_dir=str(run_dir),
            save_warmup=False,
        )

        idata = az.from_cmdstanpy(
            posterior=fit,
            log_likelihood='log_lik',
        )
        idata = _compact_idata(idata)
        idata.to_netcdf(str(idata_path))

    loo = az.loo(idata, pointwise=False)
    loo_dict[name] = loo
    elpd_val, se_val = _extract_elpd_and_se(loo)

    summ = az.summary(idata, var_names=core_posterior_vars)
    summary_dict[name] = summ

    results.append({
        'name': name,
        'sigma_alpha': cfg['sigma_alpha'],
        'sigma_prov': cfg['sigma_prov'],
        'sigma_beta_log_fire_size': cfg['sigma_beta_log_fire_size'],
        'sigma_beta_fn': cfg['sigma_beta_fn'],
        'elpd': elpd_val,
        'elpd_se': se_val,
        'alpha_mean': float(idata.posterior['alpha'].values.mean()),
        'beta_log_fire_size_mean': float(idata.posterior['beta_log_fire_size'].values.mean()),
        'beta_fn_mean': float(idata.posterior['beta_fn'].values.mean()),
    })

results_df = pd.DataFrame(results).sort_values('name').reset_index(drop=True)
results_df.to_csv(out_root / 'results_summary.csv', index=False)
results_df

  0%|          | 0/3 [00:00<?, ?it/s]

Running half_all


21:49:23 - cmdstanpy - INFO - CmdStan start processing
















































































































































































































































































































































































































chain 1: 100%|██████████| 1500/1500 [01:55<00:00, 13.01it/s, (Sampling completed)]





chain 2: 100%|██████████| 1500/1500 [01:55<00:00, 13.01it/s, (Sampling completed)]






chain 3: 100%|██████████| 1500/1500 [01:55<00:00, 13.01it/s, (Sampling completed)]







chain 4: 100%|██████████| 1500/1500 [01:55<00:00, 13.01it/s, (Sampling completed)]


21:51:18 - cmdstanpy - INFO - CmdStan done processing.


 33%|███▎      | 1/3 [02:34<05:09, 154.95s/it]21:51:57 - cmdstanpy - INFO - CmdStan start processing


...

Running double_all













































































































































































































































































































































































































chain 1: 100%|██████████| 1500/1500 [06:07<00:00,  4.08it/s, (Sampling completed)]





chain 2: 100%|██████████| 1500/1500 [06:07<00:00,  4.08it/s, (Sampling completed)]






chain 3: 100%|██████████| 1500/1500 [06:07<00:00,  4.08it/s, (Sampling completed)]







chain 4: 100%|██████████| 1500/1500 [06:07<00:00,  4.08it/s, (Sampling completed)]


22:02:11 - cmdstanpy - INFO - CmdStan done processing.


100%|██████████| 3/3 [13:23<00:00, 267.80s/it]


,name,sigma_alpha,sigma_prov,sigma_beta_log_fire_size,sigma_beta_fn,elpd,elpd_se,alpha_mean,beta_log_fire_size_mean,beta_fn_mean
0,baseline,2.0,1.0,1.0,1.0,-2019.054859,66.511091,-4.058001,0.485617,1.213046
1,double_all,4.0,2.0,2.0,2.0,-2018.053110,66.686349,-4.347124,0.489837,1.257773
2,half_all,1.0,0.5,0.5,0.5,-2023.379318,66.182556,-3.805307,0.473014,1.086243


In [ ]:
# Compare ELPD across prior settings
comp_sensitivity = (
    results_df[['name', 'elpd', 'elpd_se']]
    .assign(delta_vs_best=lambda d: d['elpd'] - d['elpd'].max())
    .sort_values('elpd', ascending=False)
    .reset_index(drop=True)
)
comp_sensitivity

In [ ]:
baseline_row = results_df[results_df['name'] == 'baseline'].iloc[0]

stability = results_df.copy()
stability['delta_elpd'] = stability['elpd'] - baseline_row['elpd']
stability['delta_alpha'] = stability['alpha_mean'] - baseline_row['alpha_mean']
stability['delta_beta_log_fire_size'] = stability['beta_log_fire_size_mean'] - baseline_row['beta_log_fire_size_mean']
stability['delta_beta_fn'] = stability['beta_fn_mean'] - baseline_row['beta_fn_mean']

stability = stability[[
    'name',
    'sigma_alpha', 'sigma_prov', 'sigma_beta_log_fire_size', 'sigma_beta_fn',
    'elpd', 'delta_elpd',
    'alpha_mean', 'delta_alpha',
    'beta_log_fire_size_mean', 'delta_beta_log_fire_size',
    'beta_fn_mean', 'delta_beta_fn',
]]
stability.sort_values('delta_elpd', ascending=False)

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.


NameError: name 'results_df' is not defined

In [ ]:
ax = stability.set_index('name')['delta_elpd'].sort_values().plot(kind='barh', figsize=(8, 4), color='#4c78a8')
ax.set_xlabel('ELPD difference vs baseline')
ax.set_ylabel('Prior setting')
ax.set_title('Prior Sensitivity: ELPD differences')
plt.tight_layout()
plt.show()

param_deltas = stability.set_index('name')[['delta_alpha', 'delta_beta_log_fire_size', 'delta_beta_fn']]
ax = param_deltas.plot(kind='bar', figsize=(10, 4))
ax.set_title('Prior Sensitivity: parameter mean shifts vs baseline')
ax.set_ylabel('Difference from baseline mean')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()